In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

C:\Users\xa03a\AppData\Local\Temp\ipykernel_22552\3588720268.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [3]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")

ENV VAR LOADED


In [4]:
DATA_FILE_PATH = os.path.join("data","hr_policy.txt")

loader = TextLoader(DATA_FILE_PATH,encoding="utf-8")
documents = loader.load()

print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])


Loaded file: data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2598

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


In [5]:
len(documents)

1

In [6]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [7]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [8]:
print(f"Total characters in document: {len(documents[0].page_content)}")

Total characters in document: 2598


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [10]:
len(chunks)

9

In [11]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [12]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)


EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


In [14]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks,embeddings_model)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [15]:
test_query = "How many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query,k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



In [16]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [17]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5
)
llm.model_name

'openai/gpt-oss-120b'

In [18]:
tr = llm.invoke("Hey What is the leave policy")

In [19]:
tr.content

'Sure! Here’s a quick rundown of our company’s leave policy:\n\n| Leave Type | Entitlement per Year | Notes |\n|------------|----------------------|-------|\n| **Annual (Paid) Leave** | **12\u202fdays** | Use for vacation, personal time, or any non‑work‑related purpose. Must be requested through the HR portal and approved by your manager. |\n| **Sick Leave** | **8\u202fdays** | For personal illness or medical appointments. A medical certificate may be required for absences of 3\u202f+ consecutive days. |\n| **Compassionate (Bereavement) Leave** | **5\u202fdays** | Granted for the death of an immediate family member or other serious personal emergencies. |\n| **Maternity Leave** | **5\u202fdays** (paid) | Available to expectant mothers. Additional statutory maternity benefits (e.g., extended unpaid leave) are provided as per local labor law. |\n| **Paternity Leave** | **5\u202fdays** (paid) | For new fathers to support the family after birth or adoption. |\n| **Study Leave** | **5\u202f

In [20]:
from langchain.agents import create_agent

In [21]:
hr_assistant = create_agent(
    model = llm,
    tools = [search_hr_policy],
    system_prompt="""
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    
    
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [22]:
def ask_hr_assistant(question:str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)
    
    response = hr_assistant.invoke({"messages":[{"role":"user","content":question}]})
    answer = response["messages"][-1].content
    
    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer
    

In [24]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"tell me which org you work for"
            }
        ]
    }
)

In [25]:
response

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='04168782-2ed2-456d-b940-e7490f69b2a4'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer user question: "tell me which org you work for". As HR assistant, we should answer that we work for Acme Crop. We should follow developer instruction: use search_hr_policy tool to look up facts before answering. The question is about which organization we work for, which is given in the system prompt: "You are a friendly HR assistant working for Acme Crop." That is not a policy document, but we could still search? The instruction says always use search_hr_policy to look up facts before answering. The fact is not in HR policy doc; it\'s in the prompt. But we could still search for "Acme Crop" maybe? The policy doc might contain organization name. We can try search.', 'tool_calls': [{'id': 'fc_2f562d83-775f-412c-bf33-6299181891fc', 'function': {'argume

In [26]:
response["messages"][-2].content

'COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'

In [27]:
print(response["messages"][-1].content)

I’m the HR assistant for **Acme Corp**.
